In [ ]:
!pip install transformers datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 11.1 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2024.12.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is

In [ ]:
# Import libraries
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from datasets import Dataset
import pandas as pd
import numpy as np
from tqdm import tqdm

In [ ]:
# Load Tiny-GPT2 model
model_name = "sshleifer/tiny-gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

print(" Tiny-GPT2 model loaded successfully!")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.51M [00:00<?, ?B/s]

✅ Tiny-GPT2 model loaded successfully!


In [ ]:
train_df = pd.read_csv("/content/mmlu_subset_12k.csv")
test_df = pd.read_csv("/content/human_ranked.csv")

print(f"Training samples: {len(train_df)}, Test samples: {len(test_df)}")

Training samples: 2000, Test samples: 480


In [ ]:
def prepare_train_examples(example):
    prompt = f"Question: {example['question']}\nA. {example['choice_0']}\nB. {example['choice_1']}\nC. {example['choice_2']}\nD. {example['choice_3']}\nChoose the correct answer:"
    index_to_letter = {0: 'A', 1: 'B', 2: 'C', 3: 'D'}
    label = index_to_letter[example['answer']]

    inputs = tokenizer(prompt, truncation=True, padding="max_length", max_length=256)
    labels = [-100] * len(inputs.input_ids)
    label_token_id = tokenizer(label, add_special_tokens=False)['input_ids'][0]
    labels[-1] = label_token_id

    return {
        'input_ids': inputs['input_ids'],
        'attention_mask': inputs['attention_mask'],
        'labels': labels
    }

In [ ]:
train_dataset = Dataset.from_pandas(train_df)
train_dataset = train_dataset.map(prepare_train_examples)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
training_args = TrainingArguments(
    output_dir="./fine_tuned_tinygpt2",
    num_train_epochs=0.2,
    per_device_train_batch_size=4,
    learning_rate=3e-5,
    weight_decay=0.01,
    logging_dir="./logs_tinygpt2",
    logging_steps=50,
    save_strategy="no",
    #evaluation_strategy="no",
    report_to="none",
    fp16=True
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    tokenizer=tokenizer
)


<ipython-input-8-8a3bc22b4849>:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
print("\n=== Fine-tuning Tiny-GPT2 Started ===")
trainer.train()

# Save the model
model.save_pretrained("./fine_tuned_tinygpt2")
tokenizer.save_pretrained("./fine_tuned_tinygpt2")
print(" Fine-tuning completed!")


=== Fine-tuning Tiny-GPT2 Started ===


`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
50,10.827500
100,10.825500


✅ Fine-tuning completed!


In [ ]:
# Reload model (optional safety)
tokenizer = AutoTokenizer.from_pretrained("./fine_tuned_tinygpt2")
model = AutoModelForCausalLM.from_pretrained("./fine_tuned_tinygpt2")
model.eval()
model = model.to(device)


In [ ]:
def get_logits(model, tokenizer, question, choices):
    prompt = f"Question: {question}\n" + "\n".join([f"{chr(65+i)}. {c}" for i, c in enumerate(choices)]) + "\nChoose the correct answer:"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256).to(model.device)

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits[:, -1, :]  # Only last token
    choice_logits = []
    for letter in ['A', 'B', 'C', 'D']:
        token_id = tokenizer.encode(letter, add_special_tokens=False)[0]  # ✅ Corrected here
        choice_logits.append(logits[0, token_id].item())
    return choice_logits


In [ ]:
results = []

for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
    question = row['question']
    choices = [row['option_0'], row['option_1'], row['option_2'], row['option_3']]

    logits = get_logits(model, tokenizer, question, choices)

    predicted_choice_idx = int(np.argmax(logits))
    correct_choice_index = int(row['correct_answer']) if 'correct_answer' in row else -1

    results.append({
        'question': question,
        'model_name': "sshleifer/tiny-gpt2",
        'variant': "tiny-variant",
        'predicted_choice': predicted_choice_idx,
        'correct_choice_index': correct_choice_index,
        'logit_A': logits[0],
        'logit_B': logits[1],
        'logit_C': logits[2],
        'logit_D': logits[3],
        'option_0': choices[0],
        'option_1': choices[1],
        'option_2': choices[2],
        'option_3': choices[3]
    })

# Save Predictions
final_df = pd.DataFrame(results)
output_path = "/content/predictions_low_group_tinygpt2.csv"
final_df.to_csv(output_path, index=False)

print(f" Predictions saved at {output_path}")

100%|██████████| 480/480 [00:07<00:00, 64.72it/s]


✅ Predictions saved at /content/predictions_low_group_tinygpt2.csv


In [ ]:
# Step 1: Reload training data
train_df = pd.read_csv("/content/mmlu_subset_2k_renamed.csv")
print(f"Training samples for Variant 2: {len(train_df)}")

Training samples for Variant 2: 2000


In [ ]:

# Step 2: Reload Tiny-GPT2 model
model_name = "sshleifer/tiny-gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

In [ ]:
def prepare_train_examples(example):
    prompt = f"Question: {example['question']}\nA. {example['choice_0']}\nB. {example['choice_1']}\nC. {example['choice_2']}\nD. {example['choice_3']}\nChoose the correct answer:"
    index_to_letter = {0: 'A', 1: 'B', 2: 'C', 3: 'D'}
    label = index_to_letter[example['answer']]

    inputs = tokenizer(prompt, truncation=True, padding="max_length", max_length=256)

    input_ids = inputs['input_ids']
    attention_mask = inputs['attention_mask']

    labels = [-100] * len(input_ids)
    label_token_id = tokenizer.encode(label, add_special_tokens=False)[0]
    labels[-1] = label_token_id

    return {
        'input_ids': list(input_ids),             # ✅ Explicitly make it a list
        'attention_mask': list(attention_mask),   # ✅ Explicitly make it a list
        'labels': list(labels)                    # ✅ Explicitly make it a list
    }


In [ ]:
train_dataset_variant2 = Dataset.from_pandas(train_df)
train_dataset_variant2 = train_dataset_variant2.map(prepare_train_examples)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
# Step 4: Training Arguments for Variant 2
training_args_variant2 = TrainingArguments(
    output_dir="./fine_tuned_tinygpt2_variant2",
    num_train_epochs=0.4,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_dir="./logs_tinygpt2_variant2",
    logging_steps=50,
    save_strategy="no",
    #evaluation_strategy="no",
    report_to="none",
    fp16=True
)

In [ ]:
trainer_variant2 = Trainer(
    model=model,
    args=training_args_variant2,
    train_dataset=train_dataset_variant2,
    tokenizer=tokenizer
)

<ipython-input-24-28350b6f0129>:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_variant2 = Trainer(


In [ ]:
# Step 5: Fine-tune
print("\n=== Fine-tuning Tiny-GPT2 Variant 2 Started ===")
trainer_variant2.train()


=== Fine-tuning Tiny-GPT2 Variant 2 Started ===


Step,Training Loss
50,10.826000
100,10.820800
150,10.811500
200,10.810500


TrainOutput(global_step=200, training_loss=10.817179870605468, metrics={'train_runtime': 257.0983, 'train_samples_per_second': 3.112, 'train_steps_per_second': 0.778, 'total_flos': 186777600.0, 'train_loss': 10.817179870605468, 'epoch': 0.4})

In [ ]:
# Save the model
model.save_pretrained("./fine_tuned_tinygpt2_variant2")
tokenizer.save_pretrained("./fine_tuned_tinygpt2_variant2")
print(" Fine-tuning Tiny-GPT2 Variant 2 complete!")

✅ Fine-tuning Tiny-GPT2 Variant 2 complete!


In [ ]:
# Reload test data
test_df = pd.read_csv("/content/human_ranked.csv")
print(f"Test samples for prediction: {len(test_df)}")

Test samples for prediction: 480


In [ ]:
# Reload fine-tuned model
tokenizer = AutoTokenizer.from_pretrained("./fine_tuned_tinygpt2_variant2")
model = AutoModelForCausalLM.from_pretrained("./fine_tuned_tinygpt2_variant2")
model.eval()
model = model.to(device)


In [ ]:
# Prediction helper
def get_logits(model, tokenizer, question, choices):
    prompt = f"Question: {question}\n" + "\n".join([f"{chr(65+i)}. {c}" for i, c in enumerate(choices)]) + "\nChoose the correct answer:"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256).to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits[:, -1, :]
    choice_logits = []
    for letter in ['A', 'B', 'C', 'D']:
        token_id = tokenizer.encode(letter, add_special_tokens=False)[0]
        choice_logits.append(logits[0, token_id].item())
    return choice_logits

In [ ]:
results_variant2 = []

for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
    question = row['question']
    choices = [row['option_0'], row['option_1'], row['option_2'], row['option_3']]

    logits = get_logits(model, tokenizer, question, choices)

    predicted_choice_idx = int(np.argmax(logits))
    correct_choice_index = int(row['correct_answer']) if 'correct_answer' in row else -1

    results_variant2.append({
        'question': question,
        'model_name': "sshleifer/tiny-gpt2",
        'variant': "tiny-medium",  # ✅ Marked as tiny-medium
        'predicted_choice': predicted_choice_idx,
        'correct_choice_index': correct_choice_index,
        'logit_A': logits[0],
        'logit_B': logits[1],
        'logit_C': logits[2],
        'logit_D': logits[3],
        'option_0': choices[0],
        'option_1': choices[1],
        'option_2': choices[2],
        'option_3': choices[3]
    })

100%|██████████| 480/480 [00:04<00:00, 101.67it/s]


In [ ]:
# Save Predictions
final_df_variant2 = pd.DataFrame(results_variant2)
output_path = "/content/predictions_low_group_tinygpt2_variant2.csv"
final_df_variant2.to_csv(output_path, index=False)

print(f" Predictions for Tiny-GPT2 Variant 2 saved at {output_path}")

✅ Predictions for Tiny-GPT2 Variant 2 saved at /content/predictions_low_group_tinygpt2_variant2.csv


In [ ]:
# Step 1: Reload training data
train_df = pd.read_csv("/content/mmlu_subset_2k_renamed.csv")
print(f"Training samples for Variant 3: {len(train_df)}")

Training samples for Variant 3: 2000


In [ ]:
# Step 2: Reload Tiny-GPT2 model
model_name = "sshleifer/tiny-gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

In [ ]:
# Step 3: Preprocessing (already fixed!)
def prepare_train_examples(example):
    prompt = f"Question: {example['question']}\nA. {example['choice_0']}\nB. {example['choice_1']}\nC. {example['choice_2']}\nD. {example['choice_3']}\nChoose the correct answer:"
    index_to_letter = {0: 'A', 1: 'B', 2: 'C', 3: 'D'}
    label = index_to_letter[example['answer']]

    inputs = tokenizer(prompt, truncation=True, padding="max_length", max_length=256)

    input_ids = inputs['input_ids']
    attention_mask = inputs['attention_mask']

    labels = [-100] * len(input_ids)
    label_token_id = tokenizer.encode(label, add_special_tokens=False)[0]
    labels[-1] = label_token_id

    return {
        'input_ids': list(input_ids),
        'attention_mask': list(attention_mask),
        'labels': list(labels)
    }

train_dataset_variant3 = Dataset.from_pandas(train_df)
train_dataset_variant3 = train_dataset_variant3.map(prepare_train_examples)


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
# Step 4: Training Arguments for Variant 3
training_args_variant3 = TrainingArguments(
    output_dir="./fine_tuned_tinygpt2_variant3",
    num_train_epochs=0.5,
    per_device_train_batch_size=8,   # ✅ Bigger batch
    learning_rate=1e-4,              # ✅ Higher LR
    weight_decay=0.01,
    logging_dir="./logs_tinygpt2_variant3",
    logging_steps=50,
    save_strategy="no",
    #evaluation_strategy="no",
    report_to="none",
    fp16=True
)

In [ ]:
trainer_variant3 = Trainer(
    model=model,
    args=training_args_variant3,
    train_dataset=train_dataset_variant3,
    tokenizer=tokenizer
)

<ipython-input-37-db622f90b622>:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_variant3 = Trainer(


In [ ]:
# Step 5: Fine-tune
print("\n=== Fine-tuning Tiny-GPT2 Variant 3 Started ===")
trainer_variant3.train()


=== Fine-tuning Tiny-GPT2 Variant 3 Started ===


Step,Training Loss
50,10.822700
100,10.806100


TrainOutput(global_step=125, training_loss=10.81253662109375, metrics={'train_runtime': 295.8609, 'train_samples_per_second': 3.38, 'train_steps_per_second': 0.422, 'total_flos': 233472000.0, 'train_loss': 10.81253662109375, 'epoch': 0.5})

In [ ]:
# Save the model
model.save_pretrained("./fine_tuned_tinygpt2_variant3")
tokenizer.save_pretrained("./fine_tuned_tinygpt2_variant3")
print(" Fine-tuning Tiny-GPT2 Variant 3 complete!")

✅ Fine-tuning Tiny-GPT2 Variant 3 complete!


In [ ]:
# Reload test data
test_df = pd.read_csv("/content/human_ranked.csv")
print(f"Test samples for prediction: {len(test_df)}")

Test samples for prediction: 480


In [ ]:
# Reload fine-tuned model
tokenizer = AutoTokenizer.from_pretrained("./fine_tuned_tinygpt2_variant3")
model = AutoModelForCausalLM.from_pretrained("./fine_tuned_tinygpt2_variant3")
model.eval()
model = model.to(device)

In [ ]:
# Prediction helper
def get_logits(model, tokenizer, question, choices):
    prompt = f"Question: {question}\n" + "\n".join([f"{chr(65+i)}. {c}" for i, c in enumerate(choices)]) + "\nChoose the correct answer:"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256).to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits[:, -1, :]
    choice_logits = []
    for letter in ['A', 'B', 'C', 'D']:
        token_id = tokenizer.encode(letter, add_special_tokens=False)[0]
        choice_logits.append(logits[0, token_id].item())
    return choice_logits

In [ ]:
results_variant3 = []

for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
    question = row['question']
    choices = [row['option_0'], row['option_1'], row['option_2'], row['option_3']]

    logits = get_logits(model, tokenizer, question, choices)

    predicted_choice_idx = int(np.argmax(logits))
    correct_choice_index = int(row['correct_answer']) if 'correct_answer' in row else -1

    results_variant3.append({
        'question': question,
        'model_name': "sshleifer/tiny-gpt2",
        'variant': "tiny-high",  # ✅ Marked this as new variant
        'predicted_choice': predicted_choice_idx,
        'correct_choice_index': correct_choice_index,
        'logit_A': logits[0],
        'logit_B': logits[1],
        'logit_C': logits[2],
        'logit_D': logits[3],
        'option_0': choices[0],
        'option_1': choices[1],
        'option_2': choices[2],
        'option_3': choices[3]
    })


100%|██████████| 480/480 [00:03<00:00, 147.54it/s]


In [ ]:
# Save Predictions
final_df_variant3 = pd.DataFrame(results_variant3)
output_path = "/content/predictions_low_group_tinygpt2_variant3.csv"
final_df_variant3.to_csv(output_path, index=False)

print(f" Predictions for Tiny-GPT2 Variant 3 saved at {output_path}")

✅ Predictions for Tiny-GPT2 Variant 3 saved at /content/predictions_low_group_tinygpt2_variant3.csv
